# Rocket Parameters Extraction

**Notebook 2a** - Data Cleaning Phase: Rocket Parameters

## Purpose
Extract and clean rocket-related parameters from raw Space Devs launch data, focusing on technical specifications and manufacturer details.

## Authors
- **Jillian Kunze** - Rocket parameter extraction and cleaning
- **Ryan Peters** - Mapping of nested parameters
- **Phillip Roman** - Cloud compatibility and hybrid data loading

## Workflow Position
1. Data Collection → Raw launch data collected via API
2. Data Cleaning → Extract parameters from raw data:
   - **2a. This Notebook** → Extract rocket parameters (15 attributes)
   - 2b. Extract launch parameters
   - 2c. Extract mission parameters
3. Data Merging → Combine all cleaned datasets

## Key Features
- **Hybrid Data Loading:** Automatically streams data from GitHub if local files are missing, ensuring compatibility with Google Colab.
- **Robust Output Generation:** Automatically checks and creates output directories (`data/cleaned data/`) to prevent file save errors.
- **Nested Data Extraction:** Parses complex JSON structures to retrieve manufacturer and configuration details.
- **Error Handling:** Manages missing keys and null values for historical launches to ensure pipeline stability.

## Key Parameters Extracted
- Rocket identification and naming
- Manufacturer information and country
- Physical specifications (length, diameter, mass)
- Performance metrics (thrust, payload capacity)
- Stage configuration and reusability

## Output
- `clean_rocket_data.tsv` - Rocket parameters ready for merging

---
**Note:** This notebook draws from testing notebooks 'DSCI511 data extraction test_v2', 'DSCI511 data extraction_rocket and launch', and 'Raw Data Filtering'

In [43]:
# from google.colab import drive
# drive.mount('/content/gdrive')

### Imports and Installs

In [44]:
from pprint import pprint
import pandas as pd
import zipfile
import json
import os
import requests
import io

## Load Raw Data

This cell implements a **hybrid loading strategy** to ensure reproducibility across different environments (Local vs. Cloud).

**Logic Flow:**
1.  **Attempt Local Load:** Checks the standard relative path (`../data/raw data/...`). If the file exists (typical for local development), it loads instantly.
2.  **Fallback to GitHub:** If the local file is missing (typical for Google Colab or fresh clones), it automatically streams the full dataset directly from the project's GitHub repository.

**Note:** This ensures the notebook runs immediately upon opening without requiring manual file uploads or drive mounting.

In [32]:
# primary option
zip_path = os.path.join("..", "data", "raw data", "raw_baseline_launches_Group7.json.zip")
launch_data_filename = 'raw_baseline_launches_Group7.json'

# cloud/colab backup
github_url = "https://github.com/Rybus07/space-legends-data/raw/main/data/raw%20data/raw_baseline_launches_Group7.json.zip"

print(f"Attempting to load data...")

try:
    print(f"Checking local path: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(launch_data_filename) as f:
            raw_launch_data = json.load(f)
    print("Success! Loaded from local file system.")

except FileNotFoundError:
    # streams from GitHub if 'Empty Drive' error in colab
    print("Local file not found (running in Cloud/Colab?).")
    print(f"Attempting download from GitHub: {github_url}")

    response = requests.get(github_url)
    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            with z.open(launch_data_filename) as f:
                raw_launch_data = json.load(f)
        print("Success! Loaded data directly from GitHub repository.")
    else:
        print(f"CRITICAL ERROR: Could not load data locally or from GitHub. Status: {response.status_code}")

Attempting to load data...
Checking local path: ../data/raw data/raw_baseline_launches_Group7.json.zip
Local file not found (running in Cloud/Colab?).
Attempting download from GitHub: https://github.com/Rybus07/space-legends-data/raw/main/data/raw%20data/raw_baseline_launches_Group7.json.zip
Success! Loaded data directly from GitHub repository.


## Data Verification & Metadata Inspection

This cell performs a **quality assurance check** on the loaded dataset to ensure integrity before processing.

**Checks performed:**
1.  **Data Type:** Verifies the loaded object is a dictionary.
2.  **Structure:** Inspects top-level keys to confirm the JSON schema matches expectations.
3.  **Metadata:** Prints the collector name, timestamp, and total launch count to verify that the correct version of the dataset (Test vs. Production) was loaded.

In [45]:
#Check data type is correct, what are the keys, look at the metadata
print(type(raw_launch_data))
pprint(raw_launch_data.keys())
print()
print(f"Collector: {raw_launch_data['collector']}, Total launches: {raw_launch_data['total_launches']}, "
    "Collection date: {raw_launch_data['collection_date']}")
print("Total number of launches:", len(raw_launch_data['launches']))

<class 'dict'>
dict_keys(['collector', 'total_launches', 'collection_date', 'launches'])

Collector: RyanPtest, Total launches: 7336, Collection date: {raw_launch_data['collection_date']}
Total number of launches: 7336


## Define Data Source Scope

This cell isolates the list of launch records from the loaded JSON object.

**Configuration Options:**
- **Default (Production):** `data_sample = raw_launch_data["launches"]` processes every record found in the loaded file.
- **Debugging (Optional):** Uncomment the slicing line (e.g., `[:100]`) to process only a small subset. This is useful for rapidly testing changes to the extraction loop logic without waiting for the full dataset to process.

In [46]:
#Save a smaller data sample to test:
#data_sample = raw_launch_data["launches"][:100]  #[-100:]

#If you want the full data instead, set the data_sample to this:
data_sample = raw_launch_data["launches"]

## Extract Rocket Parameters

This cell iterates through the launch records to extract detailed technical specifications for each rocket.

**Key Extraction Logic:**
1.  **Nested Data Retrieval:** Safely navigates deep JSON structures to retrieve manufacturer details (Name, Country, Type), adding error handling to prevent crashes when manufacturer data is missing.
2.  **Payload Aggregation:** Combines multiple payload capacity metrics (LEO, GTO, GEO, SSO) into a single descriptive string for easier analysis.
3.  **Data Structuring:** Compiles all 15 attributes into a structured list (`rocket_table`), ensuring the `ID` is captured to allow merging with the Launch and Mission datasets.

In [47]:
rocket_table = [] #intialize

#Headers for the data we will gather in loop:
rocket_header = ["ID", "Rocket_name", "Manufacturer_name", "Manufacturer_country", "Manufacturer_company_type",
                 "Reusability", "Min_no_stages", "Max_no_stages", "Rocket_length", "Rocket_diameter", "Launch_cost", "Liftoff_mass_tons",
                 "Liftoff_thrust_kN", "Rocket_apogee", "Payload_mass"]

for launch in data_sample:
    ID = launch['id']

    rocket_name = launch['rocket']['configuration']['full_name']

    #The below sometimes causes 'list index out of range' errors - adding code to fill with None in those cases
    if len(launch['rocket']['configuration']['families']) > 0:
        manuf_name = launch['rocket']['configuration']['families'][0]['manufacturer'][0]['name']
        manuf_country = launch['rocket']['configuration']['families'][0]['manufacturer'][0]['country'][0]['name']
        manuf_type = launch['rocket']['configuration']['families'][0]['manufacturer'][0]['type']['name']
    else:
        manuf_name = None
        manuf_country = None
        manuf_type = None

    #provider = launch["launch_service_provider"]["name"] #this was included in Mission dataset
    reuse = launch['rocket']['configuration']['reusable']
    min_stage = launch['rocket']['configuration']['min_stage']
    max_stage = launch['rocket']['configuration']['max_stage']
    length = launch['rocket']['configuration']['length']
    diameter = launch['rocket']['configuration']['diameter']
    cost = launch['rocket']['configuration']['launch_cost']
    mass = launch['rocket']['configuration']['launch_mass']
    thrust = launch['rocket']['configuration']['to_thrust']
    apogee = launch['rocket']['configuration']['apogee']

    #Below, we'll combine information from multiple fields to fill out the 'payload' column
    LEO_payload = launch['rocket']['configuration']['leo_capacity'] #rocket payload mass to LEO (kg)
    GTO_payload = launch['rocket']['configuration']['gto_capacity'] #rocket payload mass to GTO (kg)
    GEO_payload = launch['rocket']['configuration']['geo_capacity'] #rocket payload mass to GEO (kg)
    SSO_payload = launch['rocket']['configuration']['sso_capacity'] #rocket payload mass to SSO (kg)

    if LEO_payload or GTO_payload or GEO_payload or SSO_payload:
        payload = "" #initialize a string
        if LEO_payload:
            payload += f"LEO {LEO_payload} "
        if GTO_payload:
            payload += f"GTO {GTO_payload} "
        if GEO_payload:
            payload += f"GEO {GEO_payload} "
        if SSO_payload:
            payload += f"SSO {SSO_payload} "
    else:
        payload = None

    row = [ID, rocket_name, manuf_name, manuf_country, manuf_type, reuse,
           min_stage, max_stage, length, diameter, cost, mass, thrust, apogee, payload]
    rocket_table.append(row)

## Create and Inspect DataFrame

This cell converts the extracted list of rocket data into a structured **pandas DataFrame** and displays the final rows for verification.

**Key Actions:**
1.  **DataFrame Creation:** Initializes `rocket_dataframe` using the collected `rocket_table` data and the defined `rocket_header` column names.
2.  **Data Verification:** Calls `.tail()` to print the last 5 rows, allowing for a quick visual check to ensure the data was populated correctly and the extraction loop finished as expected.

In [48]:
rocket_dataframe = pd.DataFrame(rocket_table, columns = rocket_header)
rocket_dataframe.tail()

,ID,Rocket_name,Manufacturer_name,Manufacturer_country,Manufacturer_company_type,Reusability,Min_no_stages,Max_no_stages,Rocket_length,Rocket_diameter,Launch_cost,Liftoff_mass_tons,Liftoff_thrust_kN,Rocket_apogee,Payload_mass
7331,5fc609f3-4aee-4ffd-968d-b4cddcd9e381,Long March 7A,China Aerospace Science and Technology Corpora...,China,Government,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7332,7afcacb9-32aa-41ee-a000-0c7158f324c2,Ariane 62,Arianespace,France,Commercial,False,2.0,2.0,63.0,5.40,85000000.0,530.0,10370.0,NaN,LEO 10350.0 GTO 5000.0 SSO 6450.0
7333,5d816773-89cb-48b9-9bf0-a9c4d8236785,Electron,None,None,None,False,2.0,3.0,18.0,1.20,6000000.0,13.0,162.0,NaN,LEO 300.0 SSO 225.0
7334,9dd2d2b7-302b-4e9e-804c-aa176f606b6f,Falcon 9 Block 5,SpaceX,United States of America,Commercial,True,1.0,2.0,70.0,3.65,52000000.0,549.0,7607.0,200.0,LEO 22800.0 GTO 8300.0
7335,6602c88f-cbff-4495-b417-a184ddb0a426,Falcon 9 Block 5,SpaceX,United States of America,Commercial,True,1.0,2.0,70.0,3.65,52000000.0,549.0,7607.0,200.0,LEO 22800.0 GTO 8300.0


## Inspect DataFrame Structure

This cell calls `.info()` to generate a concise summary of the `rocket_dataframe`.

**Key Insights:**
1.  **Data Types:** Verifies that numerical columns (like `Rocket_length`, `Liftoff_mass_tons`) are correctly cast as floats, while categorical data remains as objects.
2.  **Missing Data:** The "Non-Null Count" column highlights attributes with significant gaps (e.g., `Launch_cost` and `Rocket_apogee`), which is critical context for any downstream analysis.
3.  **Memory Usage:** Confirms the dataset size is manageable within standard memory limits.

In [49]:
rocket_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7336 entries, 0 to 7335
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         7336 non-null   object 
 1   Rocket_name                7336 non-null   object 
 2   Manufacturer_name          6677 non-null   object 
 3   Manufacturer_country       6677 non-null   object 
 4   Manufacturer_company_type  6677 non-null   object 
 5   Reusability                7336 non-null   bool   
 6   Min_no_stages              5914 non-null   float64
 7   Max_no_stages              5914 non-null   float64
 8   Rocket_length              5800 non-null   float64
 9   Rocket_diameter            5799 non-null   float64
 10  Launch_cost                2101 non-null   float64
 11  Liftoff_mass_tons          5702 non-null   float64
 12  Liftoff_thrust_kN          4160 non-null   float64
 13  Rocket_apogee              1348 non-null   float

## Analyze Manufacturer Distribution

This cell calculates the frequency of each unique value in the `Manufacturer_name` column using `.value_counts()`.

**Key Insights:**
1.  **Dominant Players:** Identifies which organizations (e.g., McDonnell Douglas, Soviet Space Program, Lockheed Martin) are most frequently represented in the current dataset.
2.  **Historical Context:** Highlights the heavy presence of legacy manufacturers and state-run programs, reflecting the historical nature of the launch data.

In [50]:
rocket_dataframe['Manufacturer_name'].value_counts()

,count
Manufacturer_name,
Progress Rocket Space Center,1029
Lockheed Martin,665
Soviet Space Program,632
Strategic Rocket Forces,621
China Aerospace Science and Technology Corporation,609
SpaceX,596
Khrunichev State Research and Production Space Center,440
Yuzhnoye Design Bureau,342
McDonnell Douglas,330


## Analyze Manufacturer Sector Distribution

This cell examines the breakdown of manufacturers by company type using `.value_counts()`.

**Key Insights:**
1.  **Industry Landscape:** Reveals the ratio between government-led space programs (e.g., NASA, Roscosmos) and the commercial space sector (e.g., SpaceX, Rocket Lab).
2.  **Category Validation:** Verifies that the classification labels (Commercial, Government, etc.) are consistent across the dataset.

In [51]:
rocket_dataframe['Manufacturer_company_type'].value_counts()

,count
Manufacturer_company_type,
Commercial,3986
Government,2689
Private,2


## Validate Data Types

This cell displays the data type (`dtype`) for each column in the `rocket_dataframe`.

**Key Insights:**
1.  **Numerical Validation:** Confirms that quantitative metrics like `Rocket_length`, `Launch_cost`, and `Liftoff_thrust_kN` are correctly stored as `float64`, allowing for statistical analysis.
2.  **String Identification:** Notes that `Payload_mass` is stored as an `object` (string) rather than a number. This is expected behavior, as this column aggregates multiple payload configurations (e.g., "LEO 1000kg GTO 500kg") into a single descriptive text field.

In [52]:
rocket_dataframe.dtypes

,0
ID,object
Rocket_name,object
Manufacturer_name,object
Manufacturer_country,object
Manufacturer_company_type,object
Reusability,bool
Min_no_stages,float64
Max_no_stages,float64
Rocket_length,float64
Rocket_diameter,float64


## Check for Missing Data

This cell performs a quick "sanity check" to see which columns have gaps (null values).

**How to interpret the output:**
- **False:** The column is complete (no missing data). We see this for critical identifiers like `ID` and `Rocket_name`, which is great—it means every single rocket has a name and tag.
- **True:** The column has at least one missing value. This is expected for technical specs (like `Launch_cost` or `Rocket_length`) since older historical records often lack these specific details.

In [53]:
rocket_dataframe.isnull().any()

,0
ID,False
Rocket_name,False
Manufacturer_name,True
Manufacturer_country,True
Manufacturer_company_type,True
Reusability,False
Min_no_stages,True
Max_no_stages,True
Rocket_length,True
Rocket_diameter,True


## Save Cleaned Data

This cell saves the processed dataframe to a TSV file.

**Features:**
- **Automatic Directory Creation:** Checks if the target folder (`../data/cleaned data/`) exists and creates it if missing, preventing `FileNotFoundError` in new environments.
- **TSV Format:** Saves as Tab-Separated Values to handle commas within text fields (e.g., descriptions or company names).

In [54]:
output_filename = "clean_rocket_data.tsv"
save_path = os.path.join("..", "data", "cleaned data", output_filename)

print(f"Attempting to save data to: {save_path}")

# checks if directory exists
folder_path = os.path.dirname(save_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Created missing directory structure: {folder_path}")

# then saves file
try:
    rocket_dataframe.to_csv(save_path, sep='\t', index=False)
    print(f"Success! Data saved to: {save_path}")
    print(f"File size: {os.path.getsize(save_path) / 1024:.2f} KB")
except Exception as e:
    print(f"Error saving file: {e}")

Attempting to save data to: ../data/cleaned data/clean_rocket_data.tsv
Success! Data saved to: ../data/cleaned data/clean_rocket_data.tsv
File size: 1026.06 KB
